# Reto 6: Validador de Códigos con Expresiones Regulares

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional
### Febrero - Julio 2026

---

## Contexto del Problema

Una empresa de logística necesita un sistema para **validar códigos de productos, envíos y empleados**. Actualmente, el proceso de validación es manual y propenso a errores.

Tu tarea es crear un **validador automático** usando expresiones regulares que verifique diferentes tipos de códigos según sus formatos específicos.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    SISTEMA DE VALIDACIÓN DE CÓDIGOS                     │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   ENTRADA                        PROCESAMIENTO            SALIDA        │
│   ───────                        ─────────────            ──────        │
│                                                                         │
│   ┌─────────────┐               ┌─────────────┐        ┌──────────┐    │
│   │ Código      │───────────────│ Expresiones │────────│ Válido   │    │
│   │ a validar   │               │ Regulares   │        │    o     │    │
│   └─────────────┘               └─────────────┘        │ Inválido │    │
│                                        │               └──────────┘    │
│   ┌─────────────┐                     │                                │
│   │ Tipo de     │─────────────────────┘                                │
│   │ código      │                                                      │
│   └─────────────┘                                                      │
│                                                                         │
│   Tipos soportados:                                                     │
│   • Producto: ABC-1234-MX                                               │
│   • Envío: ENV-2024-03-15-001234                                        │
│   • Empleado: EMP-AAA-9999                                              │
│   • Factura: FAC-A-123456                                               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

## Especificaciones de Formatos

### 1. Código de Producto
```
Formato: ABC-1234-MX
         ───  ────  ──
          │    │    └── País: 2 letras mayúsculas
          │    └─────── Número: exactamente 4 dígitos
          └──────────── Categoría: exactamente 3 letras mayúsculas

Ejemplos válidos:   TEC-0001-MX, ALI-9999-US, ROB-1234-CA
Ejemplos inválidos: tec-0001-MX, TEC-001-MX, TEC-12345-MX, TEC-0001-M
```

### 2. Código de Envío
```
Formato: ENV-YYYY-MM-DD-NNNNNN
         ───  ────  ──  ──  ──────
          │    │    │   │    └── Número secuencial: 6 dígitos
          │    │    │   └─────── Día: 01-31
          │    │    └──────────── Mes: 01-12
          │    └───────────────── Año: 4 dígitos (2020-2030)
          └────────────────────── Prefijo: siempre "ENV"

Ejemplos válidos:   ENV-2024-03-15-001234, ENV-2025-12-01-999999
Ejemplos inválidos: ENV-2019-03-15-001234, ENV-2024-13-15-001234, ENV-2024-03-32-001234
```

### 3. Código de Empleado
```
Formato: EMP-XXX-NNNN
         ───  ───  ────
          │    │    └── Número: 4 dígitos (no puede empezar con 0)
          │    └─────── Departamento: 3 letras mayúsculas
          └──────────── Prefijo: siempre "EMP"

Departamentos válidos: VEN (Ventas), ADM (Administración), TEC (Tecnología),
                       LOG (Logística), RHH (Recursos Humanos)

Ejemplos válidos:   EMP-VEN-1234, EMP-TEC-9999, EMP-ADM-1000
Ejemplos inválidos: EMP-VEN-0123, EMP-XXX-1234, EMP-VEN-123
```

### 4. Código de Factura
```
Formato: FAC-S-NNNNNN
         ───  ─  ──────
          │   │    └── Número: 6 dígitos
          │   └─────── Serie: A, B, C, D o E
          └──────────── Prefijo: siempre "FAC"

Ejemplos válidos:   FAC-A-123456, FAC-E-000001
Ejemplos inválidos: FAC-F-123456, FAC-A-12345, FAC-a-123456
```

## Requerimientos Funcionales

### Parte 1: Funciones de Validación Individual (40%)

Implementa una función para cada tipo de código:

```python
def validar_producto(codigo: str) -> dict:
    """Valida código de producto y extrae componentes."""
    # Retorna: {"valido": bool, "categoria": str, "numero": str, "pais": str}

def validar_envio(codigo: str) -> dict:
    """Valida código de envío y extrae componentes."""
    # Retorna: {"valido": bool, "fecha": str, "secuencial": str}

def validar_empleado(codigo: str) -> dict:
    """Valida código de empleado y extrae componentes."""
    # Retorna: {"valido": bool, "departamento": str, "numero": str}

def validar_factura(codigo: str) -> dict:
    """Valida código de factura y extrae componentes."""
    # Retorna: {"valido": bool, "serie": str, "numero": str}
```

### Parte 2: Validador Universal (30%)

Crea una función que detecte automáticamente el tipo de código y lo valide:

```python
def validar_codigo(codigo: str) -> dict:
    """
    Detecta el tipo de código y lo valida.
    
    Retorna:
        {
            "codigo": str,          # Código original
            "tipo": str,            # producto/envio/empleado/factura/desconocido
            "valido": bool,
            "detalles": dict        # Componentes extraídos si es válido
        }
    """
```

### Parte 3: Procesamiento por Lotes (30%)

Implementa una función que procese una lista de códigos y genere un reporte:

```python
def procesar_lote(codigos: list) -> dict:
    """
    Procesa múltiples códigos y genera estadísticas.
    
    Retorna:
        {
            "total": int,
            "validos": int,
            "invalidos": int,
            "por_tipo": {
                "producto": {"total": int, "validos": int},
                "envio": {"total": int, "validos": int},
                ...
            },
            "detalle": [lista de resultados individuales]
        }
    """
```

## Código Base

Usa este código como punto de partida:

In [1]:
import re
from typing import Dict, List

# Departamentos válidos para empleados
DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']

# Series válidas para facturas
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

In [2]:
# PARTE 1: Funciones de validación individual

def validar_producto(codigo: str) -> Dict:
    """
    Valida código de producto.
    Formato: ABC-1234-MX (3 letras - 4 dígitos - 2 letras país)
    """
    resultado = {
        "valido": False,
        "categoria": None,
        "numero": None,
        "pais": None
    }
    
    # TODO: Implementa el patrón regex y la validación
    # Usa grupos de captura para extraer: categoría, número, país
    
    return resultado

In [3]:
def validar_envio(codigo: str) -> Dict:
    """
    Valida código de envío.
    Formato: ENV-YYYY-MM-DD-NNNNNN
    - Año: 2020-2030
    - Mes: 01-12
    - Día: 01-31
    """
    resultado = {
        "valido": False,
        "fecha": None,
        "secuencial": None
    }
    
    # TODO: Implementa el patrón regex y la validación
    # Valida rangos de fecha (año 2020-2030, mes 01-12, día 01-31)
    # Nota: La validación de días por mes (ej: febrero 28/29) es opcional
    
    return resultado

In [4]:
def validar_empleado(codigo: str) -> Dict:
    """
    Valida código de empleado.
    Formato: EMP-XXX-NNNN
    - Departamento: debe estar en DEPARTAMENTOS_VALIDOS
    - Número: 4 dígitos, no empieza con 0
    """
    resultado = {
        "valido": False,
        "departamento": None,
        "numero": None
    }
    
    # TODO: Implementa el patrón regex y la validación
    # Verifica que el departamento esté en la lista de válidos
    # El número no debe empezar con 0
    
    return resultado

In [5]:
def validar_factura(codigo: str) -> Dict:
    """
    Valida código de factura.
    Formato: FAC-S-NNNNNN
    - Serie: A, B, C, D o E
    - Número: 6 dígitos
    """
    resultado = {
        "valido": False,
        "serie": None,
        "numero": None
    }
    
    # TODO: Implementa el patrón regex y la validación
    # La serie debe estar en SERIES_VALIDAS
    
    return resultado

In [6]:
# PARTE 2: Validador universal

def validar_codigo(codigo: str) -> Dict:
    """
    Detecta el tipo de código y lo valida.
    """
    resultado = {
        "codigo": codigo,
        "tipo": "desconocido",
        "valido": False,
        "detalles": {}
    }
    
    # TODO: Implementa la detección de tipo basada en el prefijo
    # y llama a la función de validación correspondiente
    
    return resultado

In [7]:
# PARTE 3: Procesamiento por lotes

def procesar_lote(codigos: List[str]) -> Dict:
    """
    Procesa múltiples códigos y genera estadísticas.
    """
    resultado = {
        "total": 0,
        "validos": 0,
        "invalidos": 0,
        "por_tipo": {
            "producto": {"total": 0, "validos": 0},
            "envio": {"total": 0, "validos": 0},
            "empleado": {"total": 0, "validos": 0},
            "factura": {"total": 0, "validos": 0},
            "desconocido": {"total": 0, "validos": 0}
        },
        "detalle": []
    }
    
    # TODO: Procesa cada código y actualiza las estadísticas
    
    return resultado

## Datos de Prueba

In [8]:
# Códigos de prueba
CODIGOS_PRUEBA = [
    # Productos
    "TEC-0001-MX",      # Válido
    "ALI-9999-US",      # Válido
    "ROB-1234-CA",      # Válido
    "tec-0001-MX",      # Inválido: minúsculas
    "TEC-001-MX",       # Inválido: solo 3 dígitos
    "TECH-0001-MX",     # Inválido: 4 letras en categoría
    
    # Envíos
    "ENV-2024-03-15-001234",    # Válido
    "ENV-2025-12-01-999999",    # Válido
    "ENV-2019-03-15-001234",    # Inválido: año fuera de rango
    "ENV-2024-13-15-001234",    # Inválido: mes 13
    "ENV-2024-03-32-001234",    # Inválido: día 32
    
    # Empleados
    "EMP-VEN-1234",     # Válido
    "EMP-TEC-9999",     # Válido
    "EMP-ADM-1000",     # Válido
    "EMP-VEN-0123",     # Inválido: empieza con 0
    "EMP-XXX-1234",     # Inválido: departamento no válido
    "EMP-VEN-123",      # Inválido: solo 3 dígitos
    
    # Facturas
    "FAC-A-123456",     # Válido
    "FAC-E-000001",     # Válido
    "FAC-B-999999",     # Válido
    "FAC-F-123456",     # Inválido: serie F no existe
    "FAC-A-12345",      # Inválido: solo 5 dígitos
    "FAC-a-123456",     # Inválido: serie en minúscula
    
    # Desconocidos
    "XXX-1234",         # Desconocido
    "RANDOM-CODE",      # Desconocido
]

## Función para Mostrar Resultados

In [9]:
def mostrar_resultado(resultado: Dict) -> None:
    """Muestra el resultado de validación de forma legible."""
    estado = "✓" if resultado["valido"] else "✗"
    print(f"{estado} {resultado['codigo']:<30} | Tipo: {resultado['tipo']:<12}")
    if resultado["valido"] and resultado["detalles"]:
        detalles = ", ".join(f"{k}: {v}" for k, v in resultado["detalles"].items() if v)
        print(f"   └── {detalles}")

In [10]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de procesamiento por lotes."""
    print("=" * 60)
    print("                 REPORTE DE VALIDACIÓN")
    print("=" * 60)
    print(f"\nTotal procesados: {reporte['total']}")
    print(f"Válidos: {reporte['validos']} ({reporte['validos']/reporte['total']*100:.1f}%)")
    print(f"Inválidos: {reporte['invalidos']} ({reporte['invalidos']/reporte['total']*100:.1f}%)")
    
    print("\nDesglose por tipo:")
    print("-" * 40)
    for tipo, stats in reporte["por_tipo"].items():
        if stats["total"] > 0:
            tasa = stats["validos"] / stats["total"] * 100
            print(f"  {tipo.capitalize():<12}: {stats['validos']:>3}/{stats['total']:<3} ({tasa:.0f}% válidos)")
    
    print("\n" + "=" * 60)

## Prueba tu Implementación

In [11]:
# Prueba de validación individual
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

# Producto
print("\n-- Productos --")
print(validar_producto("TEC-0001-MX"))
print(validar_producto("tec-0001-MX"))

# Envío
print("\n-- Envíos --")
print(validar_envio("ENV-2024-03-15-001234"))
print(validar_envio("ENV-2024-13-15-001234"))

# Empleado
print("\n-- Empleados --")
print(validar_empleado("EMP-VEN-1234"))
print(validar_empleado("EMP-VEN-0123"))

# Factura
print("\n-- Facturas --")
print(validar_factura("FAC-A-123456"))
print(validar_factura("FAC-F-123456"))

PRUEBA DE FUNCIONES INDIVIDUALES

-- Productos --
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}

-- Envíos --
{'valido': False, 'fecha': None, 'secuencial': None}
{'valido': False, 'fecha': None, 'secuencial': None}

-- Empleados --
{'valido': False, 'departamento': None, 'numero': None}
{'valido': False, 'departamento': None, 'numero': None}

-- Facturas --
{'valido': False, 'serie': None, 'numero': None}
{'valido': False, 'serie': None, 'numero': None}


In [12]:
# Prueba del validador universal
print("\nPRUEBA DE VALIDADOR UNIVERSAL")
print("=" * 50)

for codigo in CODIGOS_PRUEBA[:10]:  # Primeros 10 códigos
    resultado = validar_codigo(codigo)
    mostrar_resultado(resultado)


PRUEBA DE VALIDADOR UNIVERSAL
✗ TEC-0001-MX                    | Tipo: desconocido 
✗ ALI-9999-US                    | Tipo: desconocido 
✗ ROB-1234-CA                    | Tipo: desconocido 
✗ tec-0001-MX                    | Tipo: desconocido 
✗ TEC-001-MX                     | Tipo: desconocido 
✗ TECH-0001-MX                   | Tipo: desconocido 
✗ ENV-2024-03-15-001234          | Tipo: desconocido 
✗ ENV-2025-12-01-999999          | Tipo: desconocido 
✗ ENV-2019-03-15-001234          | Tipo: desconocido 
✗ ENV-2024-13-15-001234          | Tipo: desconocido 


In [13]:
# Prueba de procesamiento por lotes
print("\nPRUEBA DE PROCESAMIENTO POR LOTES")
reporte = procesar_lote(CODIGOS_PRUEBA)
mostrar_reporte(reporte)


PRUEBA DE PROCESAMIENTO POR LOTES
                 REPORTE DE VALIDACIÓN

Total procesados: 0


ZeroDivisionError: division by zero

## Salida Esperada

```
==============================================================
                 REPORTE DE VALIDACIÓN
==============================================================

Total procesados: 26
Válidos: 12 (46.2%)
Inválidos: 14 (53.8%)

Desglose por tipo:
----------------------------------------
  Producto    :   3/  6 (50% válidos)
  Envio       :   2/  5 (40% válidos)
  Empleado    :   3/  6 (50% válidos)
  Factura     :   3/  6 (50% válidos)
  Desconocido :   0/  2 (0% válidos)

==============================================================
```

## Bonus: Funcionalidades Extra (Opcional)

Para puntos adicionales, implementa:

### 1. Sugerencia de corrección
Si un código es inválido pero similar a uno válido, sugiere la corrección:

```python
def sugerir_correccion(codigo: str) -> str:
    # Ejemplo: "tec-0001-MX" → "TEC-0001-MX" (convertir a mayúsculas)
```

### 2. Validación de fecha real
Para envíos, valida que la fecha sea real (no 31 de febrero):

```python
def validar_fecha_real(anio: int, mes: int, dia: int) -> bool:
    # Usa el módulo datetime para validar
```

### 3. Exportar resultados a CSV
```python
def exportar_resultados(reporte: Dict, archivo: str) -> None:
    # Guarda el detalle de validación en un archivo CSV
```

In [ ]:
# BONUS: Implementa aquí las funcionalidades extra

def sugerir_correccion(codigo: str) -> str:
    """Sugiere corrección para códigos inválidos."""
    # TODO: Implementar
    pass

def validar_fecha_real(anio: int, mes: int, dia: int) -> bool:
    """Valida que una fecha sea real."""
    # TODO: Implementar
    pass

def exportar_resultados(reporte: Dict, archivo: str) -> None:
    """Exporta resultados a CSV."""
    # TODO: Implementar
    pass

---

## Criterios de Evaluación

| Criterio | Puntos |
|----------|--------|
| **Parte 1: Validadores individuales** | 40 |
| - `validar_producto()` funciona correctamente | 10 |
| - `validar_envio()` funciona correctamente | 10 |
| - `validar_empleado()` funciona correctamente | 10 |
| - `validar_factura()` funciona correctamente | 10 |
| **Parte 2: Validador universal** | 30 |
| - Detecta correctamente el tipo de código | 15 |
| - Retorna la estructura correcta | 15 |
| **Parte 3: Procesamiento por lotes** | 30 |
| - Procesa todos los códigos | 10 |
| - Genera estadísticas correctas | 10 |
| - Maneja códigos desconocidos | 10 |
| **Bonus** | +10 |
| **Total** | 100 (+10) |

---

## Entregables

1. Notebook con todas las funciones implementadas
2. Todas las celdas ejecutadas mostrando resultados
3. Comentarios explicando los patrones regex utilizados

---

## Consejos

- Usa [regex101.com](https://regex101.com/) para probar tus expresiones regulares
- Recuerda usar raw strings (`r'...'`) para los patrones
- Usa grupos de captura `()` para extraer partes del código
- Primero haz que funcione, luego optimiza